# RJMCMC
Checking data stationarity by implementing RJMCMC analysis with change points.


In [ ]:
from torch.utils.data import ConcatDataset
import glob
import xarray as xr

basin_files = glob.glob("Caravan-nc/timeseries/netcdf/camels/*.nc")
basin_files = basin_files[:25]

In [6]:
import numpy as np
from scipy.stats import pearson3

def calc_return_periods(ds):
    '''
    Takes in nc ds and treats it using B17b return period fitting 
    Returns dict (water_year, return period of max streamflow event that year)
    '''

    annual_max = ds['streamflow'].groupby('water_year').max('date')
    annual_max = annual_max.where(annual_max > 0, drop=True)  # remove zero-flow years
    log_annual_max = np.log10(annual_max)

    # annual_max = ds['streamflow'].groupby('water_year').max('date')
    # log_annual_max = np.log10(annual_max) # For the B17b fitting procedure

    params = pearson3.fit(log_annual_max)
    skew, mu, sig = params # shape location scale of the fit

    # Calculate the return years by evaluating the fit PDF distribution
    p3cdf = pearson3.cdf(log_annual_max, skew, loc=mu, scale=sig)
    return_periods_list = 1/(1-p3cdf)

    # I want these to live in a dictionary
    return_periods = dict(zip(annual_max['water_year'].values, return_periods_list))

    return return_periods

def add_water_year(ds):
    '''
    Water year is defined as Oct 1 to  Sep 30.
    Oct, Nov, Dec are labelled as regular year + 1
    '''
    t = ds['date']
    water_year = t.dt.year + (t.dt.month >= 10)
    new_ds = ds.assign_coords(water_year=water_year)
    return new_ds

In [ ]:
from torch.utils.data import ConcatDataset

multibasin_annual_maxima = [] # One entry for each basin

for basin_file in basin_files:
    ds = xr.open_dataset(basin_file)

    # Get rid of NaN values
    ds = ds.dropna(dim='date', 
               subset=['streamflow', 
                       'total_precipitation_sum',
                       'temperature_2m_mean',
                       'surface_net_solar_radiation_mean',
                       'surface_net_thermal_radiation_mean',
                       'dewpoint_temperature_2m_mean',
                       'u_component_of_wind_10m_mean',
                       'v_component_of_wind_10m_mean'], # From which features
                 how='any') # Any of the subsets

    # Water year
    ds = add_water_year(ds)

    # Put the max streamflow values into a list
    basin_annual_maxima = ds['streamflow'].groupby('water_year').max('date')
    basin_annual_maxima = basin_annual_maxima.where(basin_annual_maxima > 0, drop=True)  # remove zero-flow years
    basin_annual_maxima_dict = dict(zip(basin_annual_maxima['water_year'].values, basin_annual_maxima.values))
    multibasin_annual_maxima.append(basin_annual_maxima_dict)


In [ ]:
import numpy as np
from scipy.stats import poisson, norm, halfnorm, uniform
from scipy.special import gammaln, logsumexp
from tqdm import tqdm
from copy import deepcopy
from itertools import chain

class RJMCMC:

    def __init__(self,
                        multibasin_annual_maxima, # a dictionary for each basin; each is a dict of annual maxima 
                        k_lam = 3, k_max = 10, # the range over which k are uniform
                        gam_mu = 0.3, gam_std = 0.6,
                        sig_mu = 0, sig_std = 1,
                 ):
        self.multibasin_annual_maxima =  [list(basin_annual_maxima.values()) for basin_annual_maxima in multibasin_annual_maxima]
        self.multibasin_key_years = [list(basin_annual_maxima.keys()) for basin_annual_maxima in multibasin_annual_maxima]

        # Prior parameters
        # We don't really know very much so all are weak priors

        # Uniform prior over k params
        self.k_lam = k_lam
        self.k_max = k_max

        # Normal prior over gamma params globally given by B17c estimates
        self.gam_mu = gam_mu
        self.gam_std = gam_std

        # Half Normal prior over the sigma params because they need to be positive
        self.sig_mu = sig_mu
        self.sig_std = sig_std

        # Mu prior is uniform over total multibasin data range
        self.mu_max = max(v for d in multibasin_annual_maxima for v in d.values())
        self.mu_min = min(v for d in multibasin_annual_maxima for v in d.values())

        # Use these to define the prior distributions
        self.prior_k = poisson(self.k_lam)
        self.prior_gam = norm(self.gam_mu, self.gam_std)
        self.prior_sig = halfnorm(self.sig_mu, self.sig_std)
        self.prior_mu = uniform(self.mu_min, self.mu_max)

    def calc_c(self):
        '''
        Here we use a little trick where we give the ratio of the births and death probs to be the inv prob as the prior.
        Then this cancels the term in the acceptance ratio, saving us moves.
        We want to cap the total b_k + d_k proposal probs to 0.9 so we have to calc a constant c.
        '''
        k_vals = np.arange(0, self.k_max + 1)

        b_k = np.array([ min(1, self.prior_k.pmf(k+1)/self.prior_k.pmf(k)) 
                        if k<self.k_max else 0.0 for k in k_vals ])
        d_k = np.array([ min(1, self.prior_k.pmf(k-1)/self.prior_k.pmf(k)) 
                        if k>0 else 0.0 for k in k_vals ])
        
        self.c = 0.9/np.max(b_k + d_k)

    def log_likelihood(self,
                params,
                basin_annual_maxima
            ):
        '''
        Evaluate the likelihood of one split given a set of fit params for that segment.
        Params is a list for one single basin.
        basin_annual_maxima is the time series of extremes
        '''
        k = (len(params) - 3) // 4

        change_points = [0] + params[0:k] # The first index of each new fit
        mus = params[k:2*k+1]
        sigs= params[2*k+1:3*k+2]
        gams = params[3*k+2:4*k+3]

        # Split the stuff by change point
        segments = []
        for i in range(0, len(change_points)-1):
            start = change_points[i]
            end = change_points[i+1]
            segment = basin_annual_maxima[start:end]
            segments.append(segment)
        segments.append(basin_annual_maxima[change_points[-1]:]) # uses residual end as the new start value

        total_lnL = 0
        for idx in range(0,len(segments)): # segment lnL are additive
            segment = segments[idx]
            n = len(segment)
            LN10 = np.log(10)
            mu = mus[idx]
            sig = sigs[idx]
            gam = gams[idx]

            alpha = 4 / (gam * gam)
            beta = (sig * gam) / 2
            chi = mu - ((2*sig)/gam)

            segment_lnL = 0
            segment_lnL += -np.sum(np.log(segment))
            segment_lnL += -n*np.log(LN10 ) - n*gammaln(alpha)
            segment_lnL += (alpha-1) * np.sum(np.log(np.abs(np.log10(segment) - chi)))
            segment_lnL += -n * alpha * np.log(np.abs(beta))
            segment_lnL += (-1/beta)  * np.sum(np.log10(segment)-chi) 

            total_lnL += segment_lnL

        return total_lnL

    def transition(self,
                        state
                        ):
        '''
        MH transition to next state in the MC.
        
        '''



            